In [10]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor

warnings.filterwarnings("ignore")

base_dir = Path.cwd()
spectra_path = base_dir / "pattern.xlsx"
label_path = base_dir / "label.xlsx"
supp_path = base_dir / "Supplementary Study.xlsx"
output_path = spectra_path.with_name("RF.xlsx")
output_dir = spectra_path.parent / "rf_pls4_mild_reg_visual"
output_dir.mkdir(exist_ok=True)

spectra_df = pd.read_excel(spectra_path, header=None)
label_df = pd.read_excel(label_path, header=None)
id_col = "ID"
target_col = "HA Yield"
feature_cols = [f"Feature_{i}" for i in range(1, spectra_df.shape[1])]
spectra_df.columns = [id_col] + feature_cols
label_df = label_df.iloc[:, :2].copy()
label_df.columns = [id_col, target_col]


def normalize_id(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, np.integer)):
        return str(x)
    if isinstance(x, (float, np.floating)):
        return str(int(x)) if x.is_integer() else str(x).rstrip("0").rstrip(".")
    return str(x).strip()


spectra_df[id_col] = spectra_df[id_col].map(normalize_id)
label_df[id_col] = label_df[id_col].map(normalize_id)
label_df[target_col] = pd.to_numeric(label_df[target_col], errors="coerce")
label_check = label_df.groupby(id_col)[target_col].nunique(dropna=False)
conflict_ids = label_check[label_check > 1].index.tolist()
if conflict_ids:
    raise ValueError(f"The following IDs have multiple different HA yields, please check label.xlsx: {conflict_ids}")
label_unique = label_df.drop_duplicates(subset=[id_col], keep="first")
df = spectra_df.merge(label_unique, on=id_col, how="left")
if df[target_col].isna().any():
    missing_ids = df.loc[df[target_col].isna(), id_col].unique()
    raise ValueError(f"The following IDs are not found in label.xlsx: {missing_ids}")

X = df[feature_cols].apply(pd.to_numeric, errors="coerce")
X = X.fillna(X.mean()).fillna(0)
y = df[target_col].astype(float)

X_mean = X.mean(axis=0)
X_std = X.std(axis=0, ddof=0).replace(0, 1)
X_standard = (X - X_mean) / X_std

n_components = 4
max_components = min(X_standard.shape[0] - 1, X_standard.shape[1])
if n_components > max_components:
    raise ValueError(f"At most {max_components} PLS components can be extracted, cannot set to 4.")

pls = PLSRegression(n_components=n_components, scale=False)
pls.fit(X_standard.values, y.values.reshape(-1, 1))
X_scores = pls.x_scores_
X_reconstructed = X_scores @ pls.x_loadings_.T
ss_total = np.sum(X_standard.values ** 2)
ss_residual = np.sum((X_standard.values - X_reconstructed) ** 2)
cumulative_variance_ratio = 1 - ss_residual / ss_total

pls_cols = [f"PLS_Component_{i + 1}" for i in range(n_components)]
df_pls = pd.DataFrame(X_scores, columns=pls_cols)
df_pls.insert(0, id_col, df[id_col].values)
df_pls[target_col] = y.values
X_model = df_pls[pls_cols].astype(float)
y_model = df_pls[target_col].astype(float)
groups = df_pls[id_col].astype(str)

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(X_model, y_model, groups=groups))
X_train = X_model.iloc[train_idx]
X_test = X_model.iloc[test_idx]
y_train = y_model.iloc[train_idx]
y_test = y_model.iloc[test_idx]

best_params = {
    'n_estimators': 1000,
    'max_depth': 5,
    'max_features': 0.8,
    'min_samples_split': 2,
    'min_samples_leaf': 2,
    'max_samples': 0.8,
    'bootstrap': True,
    'ccp_alpha': 0.0
}

model = RandomForestRegressor(random_state=42, n_jobs=-1, **best_params)
model.fit(X_train, y_train)
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
train_mae = mean_absolute_error(y_train, y_train_pred)
test_mae = mean_absolute_error(y_test, y_test_pred)
train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)
train_mse = mean_squared_error(y_train, y_train_pred)
test_mse = mean_squared_error(y_test, y_test_pred)
train_mre = np.mean(np.abs(y_train - y_train_pred) / y_train) * 100
test_mre = np.mean(np.abs(y_test - y_test_pred) / y_test) * 100
gap_tr_te = abs(train_r2 - test_r2)

train_result_df = pd.DataFrame({
    id_col: df_pls.iloc[train_idx][id_col].values,
    "Dataset": "Training",
    "True HA Yield": y_train.values,
    "Predicted HA Yield": y_train_pred
})
test_result_df = pd.DataFrame({
    id_col: df_pls.iloc[test_idx][id_col].values,
    "Dataset": "Test",
    "True HA Yield": y_test.values,
    "Predicted HA Yield": y_test_pred
})
all_result_df = pd.concat([train_result_df, test_result_df], axis=0).reset_index(drop=True)


def id_sort_key(x):
    try:
        return float(str(x).strip())
    except ValueError:
        return np.inf


all_result_df["Sort Key"] = all_result_df[id_col].map(id_sort_key)
all_result_df["Group Index"] = all_result_df.groupby(id_col).cumcount() + 1
all_result_sorted_df = all_result_df.sort_values(
    by=["Sort Key", "Group Index"], ascending=[True, True]
).drop(columns=["Sort Key"])

supp_df = pd.read_excel(supp_path, header=None)
expected_cols = 1 + len(feature_cols)
if supp_df.shape[1] != expected_cols:
    raise ValueError(
        f"Supplementary Study.xlsx has {supp_df.shape[1]} columns, "
        f"expected {expected_cols} (1 label column + {len(feature_cols)} features)."
    )

supp_label = pd.to_numeric(supp_df.iloc[:, 0], errors="coerce")
supp_features = supp_df.iloc[:, 1:].copy()
supp_features.columns = feature_cols

if supp_label.isna().any():
    raise ValueError("Supplementary Study.xlsx contains non-numeric values in the first label column.")

X_supp = supp_features[feature_cols].apply(pd.to_numeric, errors="coerce")
X_supp = X_supp.fillna(X_mean).fillna(0)
X_supp_standard = (X_supp - X_mean) / X_std

X_supp_scores = pls.transform(X_supp_standard.values)
supp_pls_df = pd.DataFrame(X_supp_scores, columns=pls_cols)

supp_pred = model.predict(supp_pls_df)

ext_mse = mean_squared_error(supp_label, supp_pred)
ext_rmse = np.sqrt(ext_mse)
ext_mae = mean_absolute_error(supp_label, supp_pred)
ext_r2 = r2_score(supp_label, supp_pred)
ext_mre = np.mean(np.abs(supp_label.values - supp_pred) / supp_label.values) * 100

supp_result_df = pd.DataFrame({
    "Row_Index": np.arange(1, len(supp_df) + 1),
    "True HA Yield": supp_label.values,
    "Predicted HA Yield": supp_pred,
    "Residual": supp_label.values - supp_pred,
    "Absolute_Error": np.abs(supp_label.values - supp_pred),
    "Relative_Error_%": np.abs(supp_label.values - supp_pred) / supp_label.values * 100
})
supp_result_full_df = pd.concat([supp_result_df, supp_pls_df], axis=1)

metrics_df = pd.DataFrame({
    "Metric": [
        "PLS Components",
        "PLS Cumulative Variance Ratio",
        "Training MSE",
        "Test MSE",
        "External MSE",
        "Training RMSE",
        "Test RMSE",
        "External RMSE",
        "Training MAE",
        "Test MAE",
        "External MAE",
        "Training MRE(%)",
        "Test MRE(%)",
        "External MRE(%)",
        "Training R2",
        "Test R2",
        "External R2",
        "|Train-Test R2 gap|",
        "|Train-External R2 gap|",
        "|Test-External R2 gap|",
    ],
    "Value": [
        n_components,
        cumulative_variance_ratio,
        train_mse,
        test_mse,
        ext_mse,
        train_rmse,
        test_rmse,
        ext_rmse,
        train_mae,
        test_mae,
        ext_mae,
        train_mre,
        test_mre,
        ext_mre,
        train_r2,
        test_r2,
        ext_r2,
        gap_tr_te,
        abs(train_r2 - ext_r2),
        abs(test_r2 - ext_r2),
    ]
})
fixed_params_df = pd.DataFrame(list(best_params.items()), columns=["Parameter", "Fixed Value"])

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    df_pls.to_excel(writer, sheet_name="pls_scores_4", index=False)
    train_result_df.to_excel(writer, sheet_name="train_predictions", index=False)
    test_result_df.to_excel(writer, sheet_name="test_predictions", index=False)
    all_result_sorted_df.to_excel(writer, sheet_name="all_predictions_by_id", index=False)
    metrics_df.to_excel(writer, sheet_name="metrics", index=False)
    fixed_params_df.to_excel(writer, sheet_name="fixed_params", index=False)
    supp_result_full_df.to_excel(writer, sheet_name="external_predictions", index=False)

supp_output_path = output_dir / "Supplementary_Study_external_validation.xlsx"
with pd.ExcelWriter(supp_output_path, engine="openpyxl") as writer:
    supp_result_full_df.to_excel(writer, sheet_name="predictions", index=False)
    pd.DataFrame({
        "Metric": ["MSE", "RMSE", "MAE", "MRE_%", "R2", "N_samples"],
        "Value": [ext_mse, ext_rmse, ext_mae, ext_mre, ext_r2, len(supp_df)]
    }).to_excel(writer, sheet_name="metrics", index=False)

print("=" * 60)
print("RF Anti-overfit Tuned Fixed Parameters:")
print(best_params)
print(f"\nPLS Components: {n_components}")
print(f"PLS Cumulative Variance Ratio: {cumulative_variance_ratio:.4%}")
print("\n===== Training Set Metrics =====")
print(f"MSE: {train_mse:.4f}   RMSE: {train_rmse:.4f}   MAE: {train_mae:.4f}   MRE: {train_mre:.2f}%   R2: {train_r2:.4f}")
print("\n===== Test Set Metrics =====")
print(f"MSE: {test_mse:.4f}   RMSE: {test_rmse:.4f}   MAE: {test_mae:.4f}   MRE: {test_mre:.2f}%   R2: {test_r2:.4f}")
print("\n===== External Validation (Supplementary Study) Metrics =====")
print(f"N samples: {len(supp_df)}")
print(f"MSE : {ext_mse:.4f}")
print(f"RMSE: {ext_rmse:.4f}")
print(f"MAE : {ext_mae:.4f}")
print(f"MRE : {ext_mre:.2f}%")
print(f"R2  : {ext_r2:.4f}")
print("\n===== R2 gap check =====")
print(f"|Train-Test|     = {gap_tr_te:.4f}")
print(f"|Train-External| = {abs(train_r2 - ext_r2):.4f}")
print(f"|Test-External|  = {abs(test_r2 - ext_r2):.4f}")
print("\n===== External Predictions Preview =====")
preview_cols = [c for c in supp_result_df.columns if c not in ["Residual", "Absolute_Error", "Relative_Error_%"]]
print(supp_result_df[preview_cols].head(10).to_string(index=False))

RF Anti-overfit Tuned Fixed Parameters:
{'n_estimators': 1000, 'max_depth': 5, 'max_features': 0.8, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_samples': 0.8, 'bootstrap': True, 'ccp_alpha': 0.0}

PLS Components: 4
PLS Cumulative Variance Ratio: 97.2369%

===== Training Set Metrics =====
MSE: 0.5430   RMSE: 0.7369   MAE: 0.5174   MRE: 6.07%   R2: 0.9294

===== Test Set Metrics =====
MSE: 1.9063   RMSE: 1.3807   MAE: 1.0716   MRE: 15.25%   R2: 0.8366

===== External Validation (Supplementary Study) Metrics =====
N samples: 10
MSE : 1.1287
RMSE: 1.0624
MAE : 0.8820
MRE : 9.64%
R2  : 0.8673

===== R2 gap check =====
|Train-Test|     = 0.0928
|Train-External| = 0.0621
|Test-External|  = 0.0307

===== External Predictions Preview =====
 Row_Index  True HA Yield  Predicted HA Yield
         1          11.25           12.650020
         2          12.15           12.787476
         3          12.14           13.078590
         4          11.62           13.875007
         5          1